# 원이 얼굴 탐지 · Colab 학습
Roboflow에서 라벨링한 YOLOv8 데이터셋 ZIP을 올려 YOLO11n을 학습합니다.

**순서:** GPU 선택 → 설치 → ZIP 업로드 → 학습 → best.pt 다운로드

- 런타임 → 런타임 유형 변경 → **T4 GPU**(또는 사용 가능한 GPU)를 선택합니다.
- Roboflow는 라벨링·데이터셋 내보내기에만 사용합니다. 웹 학습은 Colab에서 합니다.
- 이미지는 원본 비율로 준비합니다. Roboflow에서 정사각형 Resize를 추가할 필요는 없습니다.
- 무료 GPU 배정은 보장되지 않습니다. GPU가 없으면 학습 셀은 실행하지 않습니다.
- 한 직캠의 인접 프레임이므로 평가 점수를 다른 영상에서의 성능으로 해석하지 않습니다.


## 1. 설치
먼저 런타임 유형을 T4 GPU로 선택하고, 각 코드 셀 왼쪽의 ▶ 버튼을 위에서부터 누릅니다.


In [ ]:
%pip install -q ultralytics==8.4.162


## 2. ZIP 업로드와 압축 풀기
PC에서 데이터셋 ZIP 이름을 **woni.zip**으로 바꾼 뒤 아래 셀에서 선택합니다. ZIP 안에는 `train`, `valid`, `test` 폴더가 있어야 합니다. 제공된 `data/woni_roboflow.zip`도 이름을 바꿔 사용할 수 있습니다.

다른 데이터로 다시 실습할 때는 새 런타임에서 시작합니다.


In [ ]:
from google.colab import files
import zipfile

# PC에서 woni.zip 파일을 선택합니다.
files.upload()

# ZIP 안의 파일을 dataset 폴더에 풉니다.
with zipfile.ZipFile("woni.zip") as zip_file:
    zip_file.extractall("/content/dataset")

# 원이 얼굴 한 종류를 학습하는 설정 파일입니다.
with open("/content/dataset/data.yaml", "w") as file:
    file.write("""path: /content/dataset
train: train/images
val: valid/images
test: test/images
names: [woni_face]
""")


## 3. GPU 학습
기존 IU 실습의 YOLO11n, 50 epochs, imgsz 640, batch 16, workers 0을 유지합니다.
`imgsz=640`은 모델 입력 크기입니다. 원본 사진을 미리 정사각형으로 저장할 필요는 없습니다.


In [ ]:
from ultralytics import YOLO

# 작은 YOLO 모델을 불러옵니다.
model = YOLO("yolo11n.pt")

# 준비한 이미지로 50번 학습합니다.
model.train(
    data="/content/dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    project="/content/runs",
    name="woni",
    exist_ok=True
)


## 4. 모델 다운로드
3번 학습이 끝난 뒤 실행합니다. 같은 런타임에서 다시 학습하면 같은 결과 폴더를 사용합니다. 이전 모델이 필요하면 재학습 전에 다운로드하세요.


In [ ]:
# 학습된 모델을 PC에 저장합니다.
files.download("/content/runs/woni/weights/best.pt")


## 5. PC 앱에 적용
1. 실행 중인 Gradio 앱을 종료합니다.
2. 내려받은 `best.pt`를 `woni_face.pt`로 이름을 바꿔 `object-detection-with-microbit/models/`에 넣습니다.
3. 프로젝트 폴더에서 아래 명령으로 앱을 시작합니다.

```powershell
.venv/Scripts/python.exe -B app.py
```

브라우저에서 http://127.0.0.1:7862 를 엽니다. 학습은 Colab, 사진·웹캠 탐지와 USB 마이크로비트 연결은 PC에서 진행합니다.

모델 다운로드를 확인한 뒤 **런타임 → 연결 해제 및 런타임 삭제**로 종료합니다. 런타임을 삭제하면 Colab 임시 파일이 사라집니다.

